# Notebook 02: SFT Trials (5 runs)

**Purpose:** Run 5 Supervised Fine-Tuning trials on `open-r1/codeforces-cots`, varying LoRA rank/modules and training hyperparameters. Evaluate each on the 10 test prompts. Pick the winner.

**Inputs:**
- `data/test_prompts.json` (10 prompts)
- `data/gold_answers.json` (filled gold answers)
- `configs/sft_trials.json` (5 trial configs)
- HuggingFace dataset: `open-r1/codeforces-cots` (subset: solutions_w_editorials)

**Outputs:** `results/sft_trial_1.json` ... `results/sft_trial_5.json` + LoRA adapters in `models/sft_trial_N/`

**Runtime:** ~3-8 hours on Kaggle T4 for all 5 trials.

---

⚠ **IMPORTANT:** Each trial saves its result immediately after completion. If Kaggle session dies mid-trial, you can re-run this notebook and it will SKIP already-completed trials. Look for "✓ Skipping (already completed)" messages.


## Step 1: Setup

In [ ]:
import os
import sys
import json
import gc
import time
from pathlib import Path
import torch

KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path("/kaggle/working/daa-helper") if KAGGLE else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

# Install dependencies if on Kaggle and not already installed
if KAGGLE:
    import subprocess
    for pkg in ["transformers>=4.45.0", "peft>=0.13.0", "trl>=0.11.0",
                "bitsandbytes>=0.44.0", "accelerate>=1.0.0", "datasets>=3.0.0",
                "sacrebleu", "bert-score"]:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

from utils.io_helpers import load_json, save_trial_result, list_existing_results
from utils.evaluation import run_inference_on_prompts, evaluate_responses, free_memory

# HF login
from huggingface_hub import login
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

HF_USERNAME = "hashirilyas18"
print("Setup complete.")


In [ ]:
!pip install --upgrade transformers huggingface_hub

## Step 2: Load configs, prompts, gold answers

In [ ]:
sft_config = load_json("sft_trials.json", base_dir="configs")
prompts_data = load_json("test_prompts.json", base_dir="data")
gold_data = load_json("gold_answers.json", base_dir="data")

prompts = [p["prompt"] for p in prompts_data["prompts"]]
gold_answers = [a["gold_answer"] for a in gold_data["answers"]]
trials = sft_config["trials"]

print(f"Loaded {len(trials)} SFT trial configurations")
for t in trials:
    print(f"  - {t['name']}: rank={t['lora_r']}, target={t['target_modules']}, lr={t['learning_rate']}")


## Step 3: Load and prepare the SFT dataset

We use `open-r1/codeforces-cots` with the `solutions_w_editorials` subset. Editorials are tutorial-style explanations of algorithmic problems — exactly the writing style we want to teach.

We subsample to ~3000 examples to keep training time reasonable.


In [ ]:
from datasets import load_dataset

print("Loading open-r1/codeforces-cots (solutions_w_editorials subset)...")
raw_ds = load_dataset("open-r1/codeforces-cots", "solutions_w_editorials", split="train")
print(f"Full dataset: {len(raw_ds)} samples")
print(f"Columns: {raw_ds.column_names}")
print(f"\nSample:")
print(json.dumps({k: str(v)[:200] for k, v in raw_ds[0].items()}, indent=2))


In [ ]:
# Subsample and format for SFT
# The dataset's 'messages' column already has chat-format conversations
N_TRAIN = 3000
N_EVAL = 200

raw_ds = raw_ds.shuffle(seed=42)
train_ds = raw_ds.select(range(min(N_TRAIN, len(raw_ds))))
eval_ds = raw_ds.select(range(N_TRAIN, min(N_TRAIN + N_EVAL, len(raw_ds))))

print(f"Train: {len(train_ds)} samples")
print(f"Eval:  {len(eval_ds)} samples")

# Verify each row has 'messages'
assert "messages" in train_ds.column_names, f"Expected 'messages' column, got: {train_ds.column_names}"


## Step 4: Define the training function

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

BASE_MODEL = "TinyLlama/TinyLlama_v1.1"


def load_base_model_for_training():
    """Load BASE_MODEL with 4-bit quantization, ready for LoRA training."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        use_safetensors=False,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model)
    return model, tokenizer


def run_sft_trial(trial_config, train_dataset, eval_dataset, output_dir):
    """Run a single SFT trial. Returns (trial_dir, train_metrics)."""
    print(f"\n{'='*70}\nSFT TRIAL: {trial_config['name']}\n{'='*70}")
    print(f"Description: {trial_config['description']}")

    model, tokenizer = load_base_model_for_training()

    # LoRA config
    peft_config = LoraConfig(
        r=trial_config["lora_r"],
        lora_alpha=trial_config["lora_alpha"],
        lora_dropout=trial_config.get("lora_dropout", 0.05),
        target_modules=trial_config["target_modules"],
        bias="none",
        task_type="CAUSAL_LM",
    )

    # SFT config (TRL >= 0.11 uses SFTConfig)
    sft_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=trial_config["num_train_epochs"],
        per_device_train_batch_size=trial_config["per_device_train_batch_size"],
        gradient_accumulation_steps=trial_config["gradient_accumulation_steps"],
        per_device_eval_batch_size=trial_config["per_device_train_batch_size"],
        learning_rate=trial_config["learning_rate"],
        logging_steps=10,
        eval_strategy="steps",
        max_length=trial_config.get("max_seq_length", 1024),
        eval_steps=100,
        save_strategy="no",  # we save manually at end
        warmup_steps=5,
        lr_scheduler_type="cosine",
        bf16=True,
        optim="paged_adamw_8bit",
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataset_text_field=None,  # use messages column
        packing=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    start = time.time()
    train_result = trainer.train()
    elapsed = time.time() - start

    # Save adapter
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Final eval loss
    eval_metrics = trainer.evaluate()

    train_metrics = {
        "train_loss": train_result.training_loss,
        "eval_loss": eval_metrics.get("eval_loss"),
        "train_runtime_seconds": elapsed,
        "train_samples_per_second": train_result.metrics.get("train_samples_per_second"),
        "global_step": train_result.global_step,
    }

    # Cleanup
    del trainer, model
    free_memory()

    return output_dir, train_metrics


def evaluate_adapter(adapter_dir, prompts, gold_answers):
    """Load adapter on top of base and evaluate on test prompts."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()

    responses = run_inference_on_prompts(model, tokenizer, prompts, max_new_tokens=512, temperature=0.7)
    eval_results = evaluate_responses(responses, gold_answers)

    sample_responses = [{"prompt": p, "response": r, "gold": g}
                       for p, r, g in zip(prompts, responses, gold_answers)]

    del model, base
    free_memory()
    return eval_results, sample_responses


## Step 5: Run all 5 SFT trials

In [ ]:
existing = set(list_existing_results(stage="sft"))
print(f"Existing SFT results: {existing}")

for trial in trials:
    result_filename = f"sft_{trial['name']}.json"
    if result_filename in existing:
        print(f"\n✓ Skipping {trial['name']} (already completed)")
        continue

    output_dir = str(PROJECT_ROOT / "models" / f"sft_{trial['name']}")
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    try:
        adapter_dir, train_metrics = run_sft_trial(trial, train_ds, eval_ds, output_dir)

        print(f"\nTraining done. Evaluating on 10 test prompts...")
        eval_results, sample_responses = evaluate_adapter(adapter_dir, prompts, gold_answers)

        save_trial_result(
            trial_name=trial["name"],
            stage="sft",
            config=trial,
            eval_results=eval_results,
            train_metrics=train_metrics,
            sample_responses=sample_responses
        )

        print(f"\n✓ {trial['name']} complete.")
        print(f"  Mean BLEU:         {eval_results['aggregate']['mean_bleu']:.2f}")
        print(f"  Mean BERTScore F1: {eval_results['aggregate']['mean_bertscore_f1']:.4f}")
        print(f"  Combined score:    {eval_results['aggregate']['combined_score']:.4f}")
        print(f"  Train loss:        {train_metrics['train_loss']:.4f}")
        print(f"  Eval loss:         {train_metrics['eval_loss']:.4f}")
        print(f"  Time:              {train_metrics['train_runtime_seconds']:.0f}s")

    except Exception as e:
        print(f"\n✗ ERROR in {trial['name']}: {e}")
        import traceback
        traceback.print_exc()
        free_memory()
        continue

print("\n\nAll trials processed.")


## Step 6: Pick the winning SFT trial

In [ ]:
results = []
for trial in trials:
    try:
        r = load_json(f"sft_{trial['name']}.json", base_dir="results")
        results.append(r)
    except FileNotFoundError:
        print(f"⚠ Missing result for {trial['name']}")

# Apply assignment selection rule: highest combined score, tie-break with lower eval loss
def sort_key(r):
    score = r["evaluation"]["aggregate"]["combined_score"]
    eval_loss = r["training_metrics"].get("eval_loss") or float("inf")
    return (-score, eval_loss)

results.sort(key=sort_key)

print("="*70)
print("SFT TRIAL RANKING (best first)")
print("="*70)
print(f"{'Trial':<30} {'BLEU':>8} {'BERTScore':>10} {'Combined':>10} {'EvalLoss':>10}")
for r in results:
    name = r["trial_name"]
    agg = r["evaluation"]["aggregate"]
    eval_loss = r["training_metrics"].get("eval_loss", float("nan"))
    print(f"{name:<30} {agg['mean_bleu']:>8.2f} {agg['mean_bertscore_f1']:>10.4f} {agg['combined_score']:>10.4f} {eval_loss:>10.4f}")

winner = results[0]
print(f"\n🏆 WINNER: {winner['trial_name']}")
print(f"   Config: {winner['config']}")

# Save winner pointer
import json
with open(PROJECT_ROOT / "results" / "sft_winner.json", "w") as f:
    json.dump({"winning_trial": winner["trial_name"],
               "winning_config": winner["config"],
               "winning_metrics": winner["evaluation"]["aggregate"]}, f, indent=2)
print(f"\nSaved pointer to results/sft_winner.json")


## Step 7: Push results to GitHub + winning adapter to HF Hub

In [ ]:
import subprocess

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
GITHUB_USER  = secrets.get_secret("GITHUB_USER")

# Push results JSONs to GitHub
cmds = [
    ["git","config","--global","user.email","you@example.com"],
    ["git","config","--global","user.name", GITHUB_USER],
    ["git","-C",str(PROJECT_ROOT),"add","results/"],
    ["git","-C",str(PROJECT_ROOT),"commit","-m","sft results"],
    ["git","-C",str(PROJECT_ROOT),"push",
     f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/daa-helper.git"],
]
for cmd in cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or r.stderr)

# Push winning adapter to HF Hub (adapters are too large for GitHub)
from utils.io_helpers import push_to_hf_hub
winner_adapter_dir = str(PROJECT_ROOT / "models" / f"sft_{winner['trial_name']}")
push_to_hf_hub(
    local_dir=winner_adapter_dir,
    repo_id=f"{HF_USERNAME}/daa-helper-tinyllama-sft-winner",
    repo_type="model",
    commit_message=f"SFT winner: {winner['trial_name']}"
)

print("\n✓ Notebook 02 complete. Run Notebook 03 next.")
